In [ ]:
import torch
import matplotlib.pyplot as plt
import os
import sys

sys.path.append('.')
from problems.exact_solutions import exact_oscillator
from methods.neural_ode import train_neural_ode, rk4_step
from methods.pinn import ReconciledPINN

def train_pinn_oscillator(epochs=2500):
    model = ReconciledPINN(in_dim=1, out_dim=1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    w0, zeta = 2.0, 0.1
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        t_f = (torch.rand((400, 1)) * 10.0).requires_grad_(True)
        t_ic = torch.zeros((1, 1), requires_grad=True)
        
        z = model(t_f)
        z_t = torch.autograd.grad(z, t_f, grad_outputs=torch.ones_like(z), create_graph=True)[0]
        z_tt = torch.autograd.grad(z_t, t_f, grad_outputs=torch.ones_like(z_t), create_graph=True)[0]
        residual = z_tt + (2 * zeta * w0) * z_t + (w0**2) * z
        loss_physics = torch.mean(residual**2)
        
        z_ic = model(t_ic)
        z_t_ic = torch.autograd.grad(z_ic, t_ic, grad_outputs=torch.ones_like(z_ic), create_graph=True)[0]
        loss_ic = (z_ic - 1.0)**2 + (z_t_ic - 0.0)**2
        
        loss = loss_physics + loss_ic
        loss.backward()
        optimizer.step()
    return model

if __name__ == "__main__":
    torch.manual_seed(42)
    print("=== DECISION FRAMEWORK: KNOWN VS UNKNOWN PHYSICS ===")
    
    t_eval = torch.linspace(0, 10, 200).view(-1, 1)
    exact_trajectory = exact_oscillator(t_eval)
    
    pinn_model = train_pinn_oscillator(epochs=2500)
    
    t_train_node = t_eval[:100] 
    y_train_node = exact_trajectory[:100]
    dt = t_eval[1] - t_eval[0]
    
    node_vector_field = train_neural_ode(t_train_node, y_train_node, epochs=800)
    
    node_pred = [exact_trajectory[0]]
    for i in range(len(t_eval) - 1):
        with torch.no_grad():
            node_pred.append(rk4_step(node_vector_field, t_eval[i], node_pred[-1], dt))
    node_pred_tensor = torch.stack(node_pred)

    with torch.no_grad():
        pinn_pred = pinn_model(t_eval)
        
    plt.figure(figsize=(12, 5))
    plt.plot(t_eval.numpy(), exact_trajectory[:, 0].numpy(), 'k-', linewidth=3, label="Exact $z(t)$", alpha=0.5)
    plt.plot(t_eval.numpy(), pinn_pred.numpy(), 'r--', linewidth=2, label="PINN (Equation Only)")
    plt.plot(t_eval.numpy(), node_pred_tensor[:, 0].numpy(), 'b:', linewidth=2, label="Neural ODE (Data Only)")
    plt.axvline(x=5.0, color='gray', linestyle='-.', label="NODE Training Window Ends")
    plt.title("Method Selection by Problem Knowledge: Equation vs. Data", fontsize=14)
    plt.xlabel("Time (s)", fontsize=12)
    plt.ylabel("Displacement z(t)", fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/physics_knowledge_test.png', dpi=300)